# Elite corpus: full-PDF versus abstract cluster agreement

Compare the saved partitions selected in **Topic_analysis_PDFs_v2.ipynb** and
**Topic analysis Elite Papers Abstracts.ipynb**. Save your selection edits there first.
Only elite works present in both representations are compared (currently 282 of 292),
matched by PDF SHA-256. The larger abstract corpus is not used. No clustering is refitted:
the PDF partition remains the one fitted on the full selection, restricted to shared works.

ARI is invariant to cluster numbering: 1 means identical partitions, around 0 means
chance-level agreement, and negative values mean below-chance agreement. It measures
agreement, not which representation is better. The representations also differ in
embedding model, so this comparison does not isolate the effect of abstract versus full text.


In [ ]:
from pathlib import Path
import ast, json, sys
import pandas as pd
from IPython.display import display
from sklearn.metrics import adjusted_rand_score

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/batill').is_dir())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from batill import topic_explorer as pdf_topics
from batill import selected_abstract_topics as abstract_topics

def saved_selection(filename):
    # Read literal settings only; never execute another notebook.
    settings = {}
    for cell in json.loads((ROOT / filename).read_text())['cells']:
        if cell['cell_type'] != 'code':
            continue
        for node in ast.parse(''.join(cell['source'])).body:
            if isinstance(node, ast.Assign):
                for target in node.targets:
                    if isinstance(target, ast.Name) and target.id in {'SOURCE', 'SAVED_PARTITION'}:
                        settings[target.id] = ast.literal_eval(node.value)
    if settings.get('SOURCE', 'saved') != 'saved':
        raise ValueError(f'{filename}: select a saved partition first; comparison never refits.')
    return settings['SAVED_PARTITION']


In [ ]:
pdf_name = saved_selection('Topic_analysis_PDFs_v2.ipynb')
abstract_name = saved_selection('Topic analysis Elite Papers Abstracts.ipynb')
pdf = pdf_topics.load_context(ROOT)
abstract = abstract_topics.load_context(ROOT)
# Restrict the PDF side to text partitions, excluding citation-based assignments.
if pdf_name not in set(pdf['metrics'].solution) | {'hierarchy_15'}:
    raise ValueError('Select a full-PDF text partition in the PDF topic notebook.')
pdf_labels, _ = pdf_topics.choose_partition(pdf, source='saved', saved=pdf_name)
abstract_labels, _ = abstract_topics.choose_partition(abstract, abstract_name)
pdf_rows = pdf['papers'][['Key', 'paper_id']].assign(pdf_cluster=pdf_labels)
abstract_rows = abstract['papers'][['Key', 'paper_id']].assign(abstract_cluster=abstract_labels)
if not set(abstract_rows.Key) <= set(pdf_rows.Key):
    raise ValueError('Abstract inputs contain works outside the elite PDF corpus.')
paired = pdf_rows.merge(abstract_rows, on='Key', validate='one_to_one', suffixes=('_pdf', '_abstract'))
if not paired.paper_id_pdf.equals(paired.paper_id_abstract):
    raise ValueError('Paper IDs disagree for matched PDF identities.')
if len(paired) < 2 or (paired.pdf_cluster <= 0).any() or (paired.abstract_cluster < 0).any():
    raise ValueError('Need at least two shared works with valid assignments (abstract cluster 0 is valid).')


In [ ]:
ari = adjusted_rand_score(paired.pdf_cluster, paired.abstract_cluster)
display(pd.DataFrame([{
    'PDF partition': pdf_name, 'Abstract partition': abstract_name,
    'Shared works': len(paired), 'PDF works without abstracts': len(pdf_rows) - len(paired),
    'PDF clusters (shared works)': paired.pdf_cluster.nunique(),
    'Abstract clusters': paired.abstract_cluster.nunique(), 'ARI': ari,
}]).style.hide(axis='index').format({'ARI': '{:.4f}'}))
print('PDF-only works:', ', '.join(pdf_rows.loc[~pdf_rows.Key.isin(paired.Key), 'paper_id']))
display(pd.crosstab(paired.pdf_cluster, paired.abstract_cluster,
                    rownames=['Full-PDF cluster'], colnames=['Abstract cluster']))
